In [1]:
# 08B_classification_enhanced.ipynb
# -------------------------------------------------------
# Train classification models (Target_Cls) on enhanced datasets
# and evaluate their accuracy, precision, recall, and F1-score.

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# =====================================================
# Configuration
# =====================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_enhanced_model_ready.csv",
    "TCS": data_dir / "tcs_enhanced_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_enhanced_model_ready.csv"
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True)
}

# =====================================================
# Helper function
# =====================================================
def evaluate_classification(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

# =====================================================
# Main Processing
# =====================================================
results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")

    if not path.exists():
        print(f"  ⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded dataset shape: {df.shape}")

    target_col = "Target_Cls"
    if target_col not in df.columns:
        print(f"  ⚠️ Skipping {ticker} — Target_Cls not found.")
        continue

    # Keep numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].copy()

    # Impute missing numeric values
    imputer = SimpleImputer(strategy="mean")
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    # Separate X and y
    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col], errors="ignore")

    if X.empty or y.empty:
        print(f"  ⚠️ Skipping {ticker} — insufficient data.")
        continue

    # Split into train/test
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    # Train and evaluate each model
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        metrics = evaluate_classification(y_test, preds)
        metrics.update({"Ticker": ticker, "Model": name})
        results.append(metrics)

        print(f"  → {name}: Acc={metrics['Accuracy']:.4f}, F1={metrics['F1']:.4f}, Prec={metrics['Precision']:.4f}, Rec={metrics['Recall']:.4f}")

# =====================================================
# Save results
# =====================================================
if results:
    results_df = pd.DataFrame(results)
    save_path = results_dir / "enhanced_classification_results_final.csv"
    results_df.to_csv(save_path, index=False)
    print("\n✅ Classification completed. Results saved to:", save_path)
    display(results_df.sort_values(["Ticker", "Accuracy"], ascending=[True, False]))
else:
    print("\n⚠️ No valid results to display.")



=== Processing RELIANCE ===
  Loaded dataset shape: (1460, 25)
  → LogisticRegression: Acc=0.7534, F1=0.7805, Prec=0.7399, Rec=0.8258
  → DecisionTree: Acc=0.5308, F1=0.5861, Prec=0.5511, Rec=0.6258
  → RandomForest: Acc=0.5068, F1=0.4286, Prec=0.5567, Rec=0.3484
  → SVM: Acc=0.5308, F1=0.6935, Prec=0.5308, Rec=1.0000

=== Processing TCS ===
  Loaded dataset shape: (1460, 25)
  → LogisticRegression: Acc=0.7260, F1=0.7619, Prec=0.6667, Rec=0.8889
  → DecisionTree: Acc=0.4932, F1=0.5034, Prec=0.4870, Rec=0.5208
  → RandomForest: Acc=0.5205, F1=0.5105, Prec=0.5141, Rec=0.5069
  → SVM: Acc=0.4932, F1=0.6606, Prec=0.4932, Rec=1.0000

=== Processing HDFCBANK ===
  Loaded dataset shape: (1460, 25)
  → LogisticRegression: Acc=0.7568, F1=0.7989, Prec=0.7157, Rec=0.9038
  → DecisionTree: Acc=0.5411, F1=0.5759, Prec=0.5687, Rec=0.5833
  → RandomForest: Acc=0.5342, F1=0.5177, Prec=0.5794, Rec=0.4679
  → SVM: Acc=0.5342, F1=0.6964, Prec=0.5342, Rec=1.0000

✅ Classification completed. Results saved

,Accuracy,Precision,Recall,F1,Ticker,Model
8,0.756849,0.715736,0.903846,0.798867,HDFCBANK,LogisticRegression
9,0.541096,0.568750,0.583333,0.575949,HDFCBANK,DecisionTree
10,0.534247,0.579365,0.467949,0.517730,HDFCBANK,RandomForest
11,0.534247,0.534247,1.000000,0.696429,HDFCBANK,SVM
0,0.753425,0.739884,0.825806,0.780488,RELIANCE,LogisticRegression
1,0.530822,0.551136,0.625806,0.586103,RELIANCE,DecisionTree
3,0.530822,0.530822,1.000000,0.693512,RELIANCE,SVM
2,0.506849,0.556701,0.348387,0.428571,RELIANCE,RandomForest
4,0.726027,0.666667,0.888889,0.761905,TCS,LogisticRegression
6,0.520548,0.514085,0.506944,0.510490,TCS,RandomForest


In [2]:
import pandas as pd
from pathlib import Path

# Paths
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
results_dir = project_root / "Results"

# Load results
basic_path = results_dir / "classification_results.csv"   # previous results
enhanced_path = results_dir / "enhanced_classification_results_final.csv"

basic = pd.read_csv(basic_path)
enhanced = pd.read_csv(enhanced_path)

# Merge on Ticker + Model
comparison = pd.merge(
    basic, enhanced,
    on=["Ticker", "Model"],
    suffixes=("_basic", "_enhanced")
)

# Compute improvements (higher = better)
comparison["Δ_Accuracy"] = comparison["Accuracy_enhanced"] - comparison["Accuracy_basic"]
comparison["Δ_F1"] = comparison["F1_enhanced"] - comparison["F1_basic"]
comparison["Δ_Precision"] = comparison["Precision_enhanced"] - comparison["Precision_basic"]
comparison["Δ_Recall"] = comparison["Recall_enhanced"] - comparison["Recall_basic"]

# Display
display(comparison.sort_values(["Ticker", "Δ_Accuracy"], ascending=[True, False]))

save_path = results_dir / "classification_comparison_basic_vs_enhanced.csv"
comparison.to_csv(save_path, index=False)
print(f"\n✅ Comparison saved to: {save_path}")


,Ticker,Model,Accuracy_basic,Precision_basic,Recall_basic,F1_basic,ROC_AUC,Directional_Acc,TrainRows,TestRows,Accuracy_enhanced,Precision_enhanced,Recall_enhanced,F1_enhanced,Δ_Accuracy,Δ_F1,Δ_Precision,Δ_Recall
6,HDFCBANK,LogisticRegression,0.506849,0.535294,0.583333,0.558282,0.519702,0.506849,1168,292,0.756849,0.715736,0.903846,0.798867,0.250000,0.240585,0.180442,0.320513
7,HDFCBANK,DecisionTree,0.479452,0.513699,0.480769,0.496689,0.479355,0.479452,1168,292,0.541096,0.568750,0.583333,0.575949,0.061644,0.079261,0.055051,0.102564
8,HDFCBANK,RandomForest,0.534247,0.598039,0.391026,0.472868,0.525995,0.534247,1168,292,0.534247,0.579365,0.467949,0.517730,0.000000,0.044862,-0.018674,0.076923
0,RELIANCE,LogisticRegression,0.503425,0.526316,0.645161,0.579710,0.497763,0.503425,1168,292,0.753425,0.739884,0.825806,0.780488,0.250000,0.200778,0.213569,0.180645
1,RELIANCE,DecisionTree,0.489726,0.714286,0.064516,0.118343,0.517660,0.489726,1168,292,0.530822,0.551136,0.625806,0.586103,0.041096,0.467760,-0.163149,0.561290
2,RELIANCE,RandomForest,0.493151,0.640000,0.103226,0.177778,0.566211,0.493151,1168,292,0.506849,0.556701,0.348387,0.428571,0.013699,0.250794,-0.083299,0.245161
3,TCS,LogisticRegression,0.534247,0.525641,0.569444,0.546667,0.515907,0.534247,1168,292,0.726027,0.666667,0.888889,0.761905,0.191781,0.215238,0.141026,0.319444
5,TCS,RandomForest,0.523973,0.608696,0.097222,0.167665,0.496856,0.523973,1168,292,0.520548,0.514085,0.506944,0.510490,-0.003425,0.342825,-0.094611,0.409722
4,TCS,DecisionTree,0.520548,0.562500,0.125000,0.204545,0.515203,0.520548,1168,292,0.493151,0.487013,0.520833,0.503356,-0.027397,0.298810,-0.075487,0.395833



✅ Comparison saved to: C:\JupyterProjects\Stock_ML_Project\Results\classification_comparison_basic_vs_enhanced.csv
